# Compare reconstructions (interactive)

**Kernel:** `conda env:.conda-diffusion` to load pickles from scratch, or local `env` if files are on `/exp`.

For full interactive patch browsing, also see archived notebooks:
- `archive/notebooks/RunDDIM2DDIM_SingleExample.ipynb`
- `inference/CompareReconstructions-ICARUS.ipynb` (legacy, still useful)

This notebook loads one pickle bundle and shows original / reconstruction / saliency.


In [ ]:
from __future__ import annotations

import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))

# Edit to a produced pickle:
PKL = Path("/scratch/7DayLifetime/munjung/ICARUS/plane1_healthy_outputs/some_stem_T200_ddim2ddim.pkl")
PATCH_ORDINAL = 0

assert PKL.is_file(), PKL
raw = pickle.load(PKL.open("rb"))
patches = {k: v for k, v in raw.items() if isinstance(k, int)}
keys = sorted(patches)
i = keys[PATCH_ORDINAL]
sample = patches[i]
reco_k = next(k for k in sample if isinstance(k, str) and "-T" in k and not k.startswith("saliency") and k != "original")
sal_k = next(k for k in sample if isinstance(k, str) and k.startswith("saliency-T"))
orig = np.squeeze(sample["original"])
reco = np.squeeze(sample[reco_k])
sal = np.squeeze(sample[sal_k])
print(PKL.name, "patch", i, "keys", reco_k, sal_k, "shape", orig.shape)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, img, t, vmin, vmax in [
    (ax[0], orig, "original", -1, 1),
    (ax[1], reco, reco_k, -1, 1),
    (ax[2], sal, sal_k, -0.5, 0.5),
]:
    im = a.imshow(img, aspect="auto", origin="lower", cmap="bwr", vmin=vmin, vmax=vmax)
    a.set_title(t)
    fig.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout()
plt.show()
